# P08 · Distancia al lugar de carga más cercano

> **¿Qué proporción de las paradas tiene un lugar de carga bip! a 300 metros o menos?**

GTFS Red Movilidad y los cinco Excel públicos de lugares de carga bip!.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/cobertura_de_carga/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# Preparación: clona el repositorio y descarga los datos públicos.
# Los datos NO vienen del repositorio: se descargan del portal de datos
# abiertos, porque el laboratorio no redistribuye datos de terceros.
!git clone --depth 1 https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git lab-dtpm
%cd lab-dtpm
!pip install --quiet 'pandas>=2.2' 'numpy>=1.26' openpyxl XlsxWriter matplotlib
!python scripts/fetch_data.py gtfs-red-movilidad bip-lugares-carga


## 1 · El problema

Las paradas y los lugares de carga describen la misma ciudad, pero no comparten una
clave. La pregunta —qué proporción de las paradas tiene un lugar de carga a 300 metros—
obliga a declarar qué es una parada, cómo se mide una distancia sobre la Tierra y qué
significa «cerca».

El cruce comprende 12 497 paradas de embarque y 1 927 locales abiertos: más de 24
millones de pares. Una fórmula equivocada sigue produciendo un número razonable. Medir
grados como si fueran metros reduce la cobertura del 59,29 % al 54,69 % y asigna otro
local a 814 paradas. Además, la distancia en línea recta no es la distancia caminada;
por eso el cuaderno separa el dato calculado del supuesto de rodeo.


## 2 · La solución que se va a aplicar

Primero se fija el denominador y se audita la supuesta llave de los locales. Después se
calcula la distancia mínima con haversine por bloques, se describe su distribución y se
mide la sensibilidad al umbral y al factor de rodeo. Finalmente se confrontan dos
fórmulas y dos representaciones de coordenadas, se localizan zonas rezagadas y se
declara el límite que impide afirmar si el lugar cercano está abierto.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.cobertura_de_carga import solution as p08
import pandas as pd

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Fijar qué filas son paradas


El denominador contiene lugares donde sube gente, no estaciones, accesos ni nodos.

> ⚠️ Contar todo `stops.txt` agrega 5 989 elementos que no son paradas de embarque.


In [ ]:
resultados = p08.analizar()
resultados['paradas_por_tipo']


### Paso 2 · Comprobar la llave de los locales


Antes del cruce, se mide si `codigo` identifica un lugar de carga.

> ⚠️ En Metro el código es la línea: 143 filas comparten solo siete códigos.


In [ ]:
resultados['identificador']


### Paso 3 · Medir el vecino más cercano


Se evalúan 24 081 719 pares con haversine por bloques y se conserva el mínimo por parada.

> ⚠️ Crear la matriz completa consume memoria innecesaria; por bloques conserva el mismo mínimo.


In [ ]:
resultados['resumen']


### Paso 4 · Describir la distribución, no solo la media


Percentiles, mediana y máximo muestran una cola que la media oculta.

**En Excel:** `=PERCENTIL.INC(rango_distancia;0,5)`

Debe dar: **249,6 m de mediana**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ El máximo de 40 km pertenece al borde de cobertura de los archivos, no a una caminata plausible.


In [ ]:
resultados['distribucion']


### Paso 5 · Mostrar cuánto depende del umbral


La cobertura se publica junto a la curva de 200 a 1 000 metros.

**En Excel:** `=CONTAR.SI(rango_distancia;"<="&E2)/CONTAR(rango_distancia)`

Debe dar: **59,29 % a 300 m**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Elegir el umbral después de ver el resultado convierte el criterio en una meta móvil.


In [ ]:
resultados['cobertura_por_umbral']


### Paso 6 · Separar línea recta de distancia caminada


El factor de rodeo es un supuesto explícito y su efecto se expresa en puntos porcentuales.

**En Excel:** `=SUMAPRODUCTO(--(rango_distancia*factor<=300))/CONTAR(rango_distancia)`

Debe dar: **46,02 % con factor 1,3**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Decir 300 m caminando a partir de una línea geodésica sobrestima la cobertura.


In [ ]:
resultados['sensibilidad_rodeo']


### Paso 7 · Validar fórmula y representación


Dos controles independientes cuantifican cuánto cambia la clasificación de cada parada.

> ⚠️ Que el agregado cambie poco no significa que las mismas paradas queden cubiertas.


In [ ]:
pd.concat([resultados['formulas_comparadas'], resultados['representaciones_comparadas']], ignore_index=True)


### Paso 8 · Localizar zonas con menor cobertura


Una rejilla publicable sustituye la comuna que no existe en el GTFS.

> ⚠️ Asignar la comuna del local más cercano a la parada crea comunas falsas en los bordes.


In [ ]:
resultados['por_rejilla'].head(10)


### Paso 9 · Declarar lo que el dato no permite afirmar


Se distingue cercanía física de disponibilidad efectiva del punto de carga.

> ⚠️ Una parada cubierta espacialmente puede apuntar a un local cuyo horario no fue publicado.


In [ ]:
resultados['limite_por_horario']


## 4 · El resultado

La cobertura es 59,29 % en línea recta y 46,02 % con un rodeo supuesto de 1,3. Son dos respuestas a dos preguntas distintas; ninguna autoriza a afirmar que el local esté abierto.

## 5 · Qué queda aprendido

1. Un cruce espacial también tiene una llave: el criterio, el umbral y la fórmula declarados.
2. La sensibilidad debe acompañar al resultado cuando una decisión metodológica cambia más de diez puntos.
3. Un agregado estable puede esconder centenares de filas que cambian de clasificación.
4. La ausencia de comuna y horario limita qué recomendación es defendible.
